# Catalog Data-Quality Audit

## tl;dr

The current generated catalog is trustworthy for the assignment demo. All 1,000 listing IDs are unique; the schema matches the current model; CSV and Parquet agree; all 29 reference variants are represented; and there are no violations of the configured year, mileage, price, payload/GVW, URL-presence, or rounding rules.

Observed weight-class shares are close to their configured targets: light 43.4% vs 45%, intermediate 14.0% vs 15%, medium 16.5% vs 15%, and heavy 26.1% vs 25%. The largest absolute difference is 1.6 percentage points.

Two limitations remain: 97 listings (9.7%) intentionally have no payload because three intermediate references publish GVW but not a defensible payload; and 36 Intra V10 listings use Tata's `smalltruckstest.tatamotors.com` host rather than the canonical production host.

## Context & Methods

This is a read-only audit of `data/generated/vehicles.parquet`, `vehicles.csv`, and `vehicle_reference_catalog.csv` at listing grain (`listing_id`). It checks completeness, uniqueness, schema, cross-format agreement, domain rules, configured distributions, reference integrity, and deterministic freshness against the current generator.

### Key Assumptions

- The intended dataset is the 1,000-row deterministic demo catalog generated with seed 42.
- `payload_kg` is the only intentionally nullable listing field; GVW remains required.
- Distribution targets are sampling probabilities, so small deviations are expected in a finite random sample.
- This audit checks internal consistency and configured rules. It does not independently re-research every manufacturer specification.

## Data

### 1. Load artifacts and current generation rules

In [1]:
from collections import Counter
from datetime import UTC, datetime
from pathlib import Path
import math
import sys

import polars as pl

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "vehicle-catalog-generator" / "src"))
sys.path.insert(0, str(ROOT / "utils" / "src"))

from vehicle_catalog_generator import generator
from vehicle_catalog_generator.models import VehicleListing
from vehicle_catalog_generator.quality import validate_catalog
from vehicle_catalog_generator.reference_data import (
    CITY_WEIGHTS,
    CONDITION_PRICE_FACTORS,
    CONDITION_WEIGHTS,
    GENERATION_PARAMETERS,
    VEHICLE_REFERENCES,
    WEIGHT_CLASS_WEIGHTS,
)
from vehicle_catalog_generator.settings import settings

DATA_DIR = ROOT / "data" / "generated"
listings = pl.read_parquet(DATA_DIR / "vehicles.parquet")
csv_listings = pl.read_csv(DATA_DIR / "vehicles.csv")
reference_artifact = pl.read_csv(DATA_DIR / "vehicle_reference_catalog.csv")
rows = listings.to_dicts()
current_reference_rows = [reference.model_dump(mode="json") for reference in VEHICLE_REFERENCES]

pl.DataFrame({
    "metric": ["listing rows", "listing columns", "reference rows", "artifact modified (UTC)"],
    "value": [
        str(listings.height),
        str(listings.width),
        str(reference_artifact.height),
        datetime.fromtimestamp((DATA_DIR / "vehicles.parquet").stat().st_mtime, UTC).isoformat(),
    ],
})

metric,value
str,str
"""listing rows""","""1000"""
"""listing columns""","""18"""
"""reference rows""","""29"""
"""artifact modified (UTC)""","""2026-08-31T13:34:01.001564+00:…"


## Results

### 2. Validate schema, completeness, keys, and cross-format consistency

In [2]:
validation = validate_catalog(listings, expected_record_count=settings.data_generation.record_count)
normalized_parquet = listings.with_columns(pl.col("purpose_tags").list.join("|"))

core_checks = {
    "schema matches VehicleListing": set(listings.columns) == set(VehicleListing.model_fields),
    "unique listing IDs": listings.get_column("listing_id").n_unique() == listings.height,
    "required fields complete": all(
        count == 0 for column, count in validation["null_counts"].items() if column != "payload_kg"
    ),
    "CSV matches Parquet": normalized_parquet.select(csv_listings.columns).equals(csv_listings),
    "all URLs use HTTPS": listings.filter(~pl.col("spec_source_url").str.starts_with("https://")).height == 0,
    "all prices positive": listings.filter(pl.col("price_inr") <= 0).height == 0,
    "prices rounded to INR 5,000": listings.filter(pl.col("price_inr") % 5_000 != 0).height == 0,
    "payload less than GVW": listings.filter(
        pl.col("payload_kg").is_not_null() & (pl.col("payload_kg") >= pl.col("gvw_kg"))
    ).height == 0,
}
pl.DataFrame({"check": list(core_checks), "passed": list(core_checks.values())})

check,passed
str,bool
"""schema matches VehicleListing""",true
"""unique listing IDs""",true
"""required fields complete""",true
"""CSV matches Parquet""",true
"""all URLs use HTTPS""",true
"""all prices positive""",true
"""prices rounded to INR 5,000""",true
"""payload less than GVW""",true


### 3. Verify mileage and price-generation rules

In [3]:
def reference_key(row):
    return (
        row["make"], row["model"], row["fuel"], str(row["vehicle_category"]),
        str(row["weight_class"]), str(row["body_type"]), row["axle_count"],
        row["payload_kg"], row["gvw_kg"], tuple(row["purpose_tags"]),
    )

reference_by_key = {reference_key(row): row for row in current_reference_rows}
current_year = datetime.now(UTC).year
mileage_violations = []
price_violations = []

for row in rows:
    age = current_year - row["year"]
    mileage_low = max(
        GENERATION_PARAMETERS.minimum_km_driven,
        int(age * settings.data_generation.min_km_per_year * GENERATION_PARAMETERS.km_variance_range[0]),
    )
    mileage_high = int(
        age * settings.data_generation.max_km_per_year * GENERATION_PARAMETERS.km_variance_range[1]
    )
    if not mileage_low <= row["km_driven"] <= mileage_high:
        mileage_violations.append(row["listing_id"])

    reference = reference_by_key[reference_key(row)]
    age_factor = max(
        GENERATION_PARAMETERS.minimum_age_factor,
        (1 - GENERATION_PARAMETERS.annual_depreciation_rate) ** age,
    )
    mileage_factor = max(
        GENERATION_PARAMETERS.minimum_mileage_factor,
        1 - row["km_driven"] / GENERATION_PARAMETERS.mileage_depreciation_distance_km,
    )
    price_before_noise = (
        reference["new_vehicle_price_anchor_inr"]
        * age_factor
        * mileage_factor
        * CONDITION_PRICE_FACTORS[row["condition"]]
    )
    possible_low = price_before_noise * GENERATION_PARAMETERS.market_noise_range[0]
    possible_high = price_before_noise * GENERATION_PARAMETERS.market_noise_range[1]
    half_rounding = GENERATION_PARAMETERS.price_rounding_interval_inr / 2
    if row["price_inr"] == GENERATION_PARAMETERS.minimum_price_inr:
        possible = possible_low <= GENERATION_PARAMETERS.minimum_price_inr + half_rounding
    else:
        possible = (
            possible_low <= row["price_inr"] + half_rounding
            and possible_high >= row["price_inr"] - half_rounding
        )
    if not possible:
        price_violations.append(row["listing_id"])

pl.DataFrame({
    "rule": ["mileage within age-dependent bounds", "price compatible with formula/noise/rounding"],
    "violations": [len(mileage_violations), len(price_violations)],
})

rule,violations
str,i64
"""mileage within age-dependent b…",0
"""price compatible with formula/…",0


### 4. Compare observed and configured distributions

In [4]:
def distribution_table(field, targets):
    counts = Counter(str(row[field]) for row in rows)
    records = []
    for name, target in targets.items():
        value = str(name)
        observed = counts[value] / len(rows)
        standard_error = math.sqrt(float(target) * (1 - float(target)) / len(rows))
        records.append({
            field: value,
            "count": counts[value],
            "observed_pct": round(observed * 100, 1),
            "target_pct": round(float(target) * 100, 1),
            "difference_pp": round((observed - float(target)) * 100, 1),
            "z_score": round((observed - float(target)) / standard_error, 2),
        })
    return pl.DataFrame(records)

print("Weight class")
display(distribution_table("weight_class", WEIGHT_CLASS_WEIGHTS))
print("Condition")
display(distribution_table("condition", CONDITION_WEIGHTS))
print("City")
display(distribution_table("city", CITY_WEIGHTS))
print(
    f"Papers verified: {listings.get_column('papers_verified').mean() * 100:.1f}% "
    f"vs {settings.data_generation.papers_verified_probability * 100:.1f}% target"
)

Weight class


weight_class,count,observed_pct,target_pct,difference_pp,z_score
str,i64,f64,f64,f64,f64
"""light""",434,43.4,45.0,-1.6,-1.02
"""intermediate""",140,14.0,15.0,-1.0,-0.89
"""medium""",165,16.5,15.0,1.5,1.33
"""heavy""",261,26.1,25.0,1.1,0.8


Condition


condition,count,observed_pct,target_pct,difference_pp,z_score
str,i64,f64,f64,f64,f64
"""excellent""",180,18.0,20.0,-2.0,-1.58
"""good""",553,55.3,55.0,0.3,0.19
"""fair""",267,26.7,25.0,1.7,1.24


City


city,count,observed_pct,target_pct,difference_pp,z_score
str,i64,f64,f64,f64,f64
"""Mumbai""",152,15.2,14.0,1.2,1.09
"""Delhi""",120,12.0,13.0,-1.0,-0.94
"""Pune""",107,10.7,10.0,0.7,0.74
"""Bengaluru""",111,11.1,10.0,1.1,1.16
"""Ahmedabad""",81,8.1,9.0,-0.9,-0.99
…,…,…,…,…,…
"""Kolkata""",86,8.6,7.0,1.6,1.98
"""Surat""",44,4.4,6.0,-1.6,-2.13
"""Jaipur""",50,5.0,5.0,0.0,0.0


Papers verified: 83.1% vs 82.0% target


### 5. Check reference coverage, intermediate variety, and reproducibility

In [5]:
variant_columns = ["make", "model", "fuel", "body_type"]
listing_variants = listings.select(variant_columns).unique()
reference_variants = reference_artifact.select(variant_columns).unique()
current_generation = generator.catalog_to_dataframe(generator.generate_catalog())

intermediate_counts = (
    listings.filter(pl.col("weight_class") == "intermediate")
    .group_by(["make", "model"])
    .len(name="listings")
    .sort(["make", "model"])
)
payload_nulls = (
    listings.filter(pl.col("payload_kg").is_null())
    .group_by(["make", "model"])
    .len(name="null_payload_listings")
    .sort(["make", "model"])
)

print(f"All 29 reference variants represented: {listing_variants.join(reference_variants, on=variant_columns).height == 29}")
print(f"Generated artifact equals current seed-42 output: {listings.equals(current_generation)}")
print(f"Payload nulls: {listings.get_column('payload_kg').null_count()} / {listings.height}")
print(f"Intra V10 rows using test hostname: {listings.filter(pl.col('spec_source_url').str.contains('smalltruckstest')).height}")
display(intermediate_counts)
display(payload_nulls)

All 29 reference variants represented: True
Generated artifact equals current seed-42 output: True
Payload nulls: 97 / 1000
Intra V10 rows using test hostname: 36


make,model,listings
str,str,u32
"""Ashok Leyland""","""Partner 6 Tyre""",18
"""BharatBenz""","""1015R""",35
"""Eicher""","""Pro 2059 Plus""",31
"""Mahindra""","""Furio 10""",25
"""Tata""","""Ultra T.9""",31


make,model,null_payload_listings
str,str,u32
"""BharatBenz""","""1015R""",35
"""Eicher""","""Pro 2059 Plus""",31
"""Tata""","""Ultra T.9""",31


## Takeaways

1. **Pass — fit for the assignment demo.** No critical or high-severity internal data-quality failures were found. The artifacts are current, deterministic, structurally complete, and consistent with the generator.
2. **Distribution is behaving correctly.** Weight class, condition, city, and verification shares are within normal finite-sample variation. All five intermediate references appear, with 18–35 listings each.
3. **Medium limitation — payload coverage.** Exactly 97 listings (9.7%) have null payloads, isolated to Tata Ultra T.9, Eicher Pro 2059 Plus, and BharatBenz 1015R. This is preferable to inventing payloads, but payload-based searches will exclude those listings unless the search can fall back to GVW.
4. **Low issue — provenance hostname.** The 36 Intra V10 listings use `smalltruckstest.tatamotors.com`. The page resolves and publishes the same 2,120 kg GVW and 1,000 kg payload, but the canonical `smalltrucks.tatamotors.com/tata-intra-v10` URL is a more durable production source.
5. **No temporal conclusion.** This is a single deterministic snapshot, so there is no historical drift or freshness trend to assess.